# DB Migration: Fill Missing obs_raw from Old → New

Finds records present in the old DB but missing from the new DB (matched on the unique key `history_id, vars_id, obs_time`) and inserts them into the new DB in batches.

**Prerequisites**: the `crmp` user on the new DB must have INSERT privileges on `obs_raw`.

In [13]:
crmp_old_url = "postgresql://crmp@db.pcic.uvic.ca/crmp?keepalives=1&keepalives_idle=300&keepalives_interval=300&keepalives_count=9&passfile=/workspaces/climo-data-importer/.pgpass"
metnorth_old_url = "postgresql://metnorth@db.pcic.uvic.ca:5433/metnorth?keepalives=1&keepalives_idle=300&keepalives_interval=300&keepalives_count=9&passfile=/workspaces/climo-data-importer/.pgpass"
crmp_new_url = "postgresql+psycopg2://crmp@/crmp?host=proddb01.pcic.uvic.ca,proddb02.pcic.uvic.ca,db01.pacificclimate.org&port=5432,5432,5432&passfile=/workspaces/climo-data-importer/.pgpass&keepalives=1&keepalives_idle=300&keepalives_interval=300&keepalives_count=9"
metnorth_new_url = "postgresql+psycopg2://metnorth@/metnorth?host=proddb01.pcic.uvic.ca,proddb02.pcic.uvic.ca,db01.pacificclimate.org&port=5432,5432,5432&passfile=/workspaces/climo-data-importer/.pgpass"

import pandas as pd
from sqlalchemy import create_engine, text

engine_old = create_engine(crmp_old_url)
engine_new = create_engine(crmp_new_url)

## Verify FK alignment between DBs

Check that all `history_id` and `vars_id` values in the old DB also exist in the new DB.
If any are missing, those rows cannot be inserted without first migrating the parent records.

In [2]:
with engine_old.connect() as conn:
    old_history_ids = set(pd.read_sql(text("SELECT DISTINCT history_id FROM obs_raw"), conn)["history_id"])
    old_vars_ids    = set(pd.read_sql(text("SELECT DISTINCT vars_id FROM obs_raw"), conn)["vars_id"])

with engine_new.connect() as conn:
    new_history_ids = set(pd.read_sql(text("SELECT history_id FROM meta_history"), conn)["history_id"])
    new_vars_ids    = set(pd.read_sql(text("SELECT vars_id FROM meta_vars"), conn)["vars_id"])

missing_history = old_history_ids - new_history_ids
missing_vars    = old_vars_ids - new_vars_ids

print(f"history_ids in old but missing from new: {len(missing_history)}")
print(f"vars_ids    in old but missing from new: {len(missing_vars)}")

if missing_history:
    print(f"  Missing history_ids: {sorted(missing_history)[:20]} ...")
if missing_vars:
    print(f"  Missing vars_ids:    {sorted(missing_vars)[:20]} ...")

history_ids in old but missing from new: 0
vars_ids    in old but missing from new: 0


## Validate history_id alignment: same station in both DBs

Check that every `history_id` present in both DBs refers to the same physical station
(matching on `station_name`, `lon`, `lat`, `elev`). Any mismatches must be investigated
before migrating — inserting under a mismatched `history_id` would corrupt the data.

In [4]:
history_query = text("""
    SELECT history_id, TRIM(station_name) AS station_name, lon, lat, elev
    FROM meta_history
    ORDER BY history_id
""")


def normalize_station_name(series):
    return (
        series.fillna("")
        .astype(str)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
        .str.upper()
    )


def values_match(left, right, tolerance=1e-6):
    both_null = left.isna() & right.isna()
    close_enough = (left - right).abs().le(tolerance).fillna(False)
    return both_null | close_enough


with engine_old.connect() as conn:
    hist_old = pd.read_sql(history_query, conn)

with engine_new.connect() as conn:
    hist_new = pd.read_sql(history_query, conn)

hist_merged = hist_old.merge(hist_new, on="history_id", suffixes=("_old", "_new"))

station_name_matches = normalize_station_name(hist_merged["station_name_old"]).eq(
    normalize_station_name(hist_merged["station_name_new"])
)
lon_matches = values_match(hist_merged["lon_old"], hist_merged["lon_new"])
lat_matches = values_match(hist_merged["lat_old"], hist_merged["lat_new"])
elev_matches = values_match(hist_merged["elev_old"], hist_merged["elev_new"])

mismatches = hist_merged.loc[
    ~(station_name_matches & lon_matches & lat_matches & elev_matches)
].copy()
mismatches["lon_abs_diff"] = (mismatches["lon_old"] - mismatches["lon_new"]).abs()
mismatches["lat_abs_diff"] = (mismatches["lat_old"] - mismatches["lat_new"]).abs()
mismatches["elev_abs_diff"] = (mismatches["elev_old"] - mismatches["elev_new"]).abs()

print(f"history_ids in both DBs: {len(hist_merged):,}")
print(f"Mismatches after normalized/null-safe comparison: {len(mismatches)}")

if len(mismatches):
    display(mismatches)
    bad_history_ids = set(mismatches["history_id"])
    print(f"\nThese {len(bad_history_ids)} history_ids will be EXCLUDED from migration.")
else:
    bad_history_ids = set()
    print("All history_ids match — safe to proceed with migration.")

history_ids in both DBs: 9,758
Mismatches after normalized/null-safe comparison: 0
All history_ids match — safe to proceed with migration.


## Count missing rows per history_id

Find the volume of rows to migrate for each `history_id`, so we can process in order and track progress.

In [14]:
cutoff = pd.Timestamp.today().normalize() - pd.Timedelta(days=5)
print(f"Checking obs_raw rows with obs_time >= {cutoff}")

recent_keys_query = text("""
    SELECT history_id, vars_id, obs_time
    FROM obs_raw
    WHERE obs_time >= :cutoff
    ORDER BY history_id, vars_id, obs_time
""")

with engine_old.connect() as conn:
    old_recent_keys = pd.read_sql(recent_keys_query, conn, params={"cutoff": cutoff})

with engine_new.connect() as conn:
    new_recent_keys = pd.read_sql(recent_keys_query, conn, params={"cutoff": cutoff})

old_recent_keys["obs_time"] = pd.to_datetime(old_recent_keys["obs_time"])
new_recent_keys["obs_time"] = pd.to_datetime(new_recent_keys["obs_time"])

old_recent_set = set(zip(old_recent_keys["history_id"], old_recent_keys["vars_id"], old_recent_keys["obs_time"]))
new_recent_set = set(zip(new_recent_keys["history_id"], new_recent_keys["vars_id"], new_recent_keys["obs_time"]))

missing_in_new = sorted(old_recent_set - new_recent_set)
extra_in_new = sorted(new_recent_set - old_recent_set)

missing_in_new_df = pd.DataFrame(missing_in_new, columns=["history_id", "vars_id", "obs_time"])
extra_in_new_df = pd.DataFrame(extra_in_new, columns=["history_id", "vars_id", "obs_time"])

missing_counts = (
    missing_in_new_df.groupby("history_id").size().reset_index(name="missing_count").sort_values(["missing_count", "history_id"], ascending=[False, True])
    if not missing_in_new_df.empty
    else pd.DataFrame(columns=["history_id", "missing_count"])
)

print(f"Old DB rows in window: {len(old_recent_keys):,}")
print(f"New DB rows in window: {len(new_recent_keys):,}")
print(f"Rows present in old but missing from new: {len(missing_in_new_df):,}")
print(f"Rows present in new but missing from old: {len(extra_in_new_df):,}")

if not missing_counts.empty:
    display(missing_counts)
else:
    print("No old→new gaps found in this window.")

Checking obs_raw rows with obs_time >= 2026-04-22 00:00:00
Old DB rows in window: 795,294
New DB rows in window: 796,322
Rows present in old but missing from new: 0
Rows present in new but missing from old: 1,028
No old→new gaps found in this window.


## Migrate recent missing rows

Fetch the missing `obs_raw` rows from the old DB for the recent window and insert them into the new DB in batches. The cell defaults to preview mode; set `EXECUTE_MIGRATION = True` to perform the insert.

In [7]:
BATCH_SIZE = 10_000
EXECUTE_MIGRATION = True

insert_sql = text("""
    INSERT INTO obs_raw (obs_time, datum, vars_id, history_id, mod_user)
    VALUES (:obs_time, :datum, :vars_id, :history_id, :mod_user)
    ON CONFLICT (history_id, vars_id, obs_time) DO NOTHING
""")

recent_rows_query = text("""
    SELECT obs_time, datum, vars_id, history_id, mod_user
    FROM obs_raw
    WHERE obs_time >= :cutoff
    ORDER BY history_id, vars_id, obs_time
""")

if missing_in_new_df.empty:
    rows_to_insert = pd.DataFrame(columns=["obs_time", "datum", "vars_id", "history_id", "mod_user"])
    print("No recent rows need migration.")
else:
    with engine_old.connect() as conn:
        old_recent_rows = pd.read_sql(recent_rows_query, conn, params={"cutoff": cutoff})

    old_recent_rows["obs_time"] = pd.to_datetime(old_recent_rows["obs_time"])
    rows_to_insert = old_recent_rows.merge(
        missing_in_new_df,
        on=["history_id", "vars_id", "obs_time"],
        how="inner",
    ).sort_values(["history_id", "vars_id", "obs_time"])

    print(f"Rows queued for insert: {len(rows_to_insert):,}")
    print(f"History IDs affected:   {rows_to_insert['history_id'].nunique():,}")
    display(rows_to_insert.head(20))

if EXECUTE_MIGRATION and not rows_to_insert.empty:
    total_inserted = 0
    with engine_new.begin() as conn_new:
        for start in range(0, len(rows_to_insert), BATCH_SIZE):
            batch = rows_to_insert.iloc[start : start + BATCH_SIZE]
            conn_new.execute(insert_sql, batch.to_dict(orient="records"))
            total_inserted += len(batch)
            print(
                f"Inserted {total_inserted:,}/{len(rows_to_insert):,} recent missing rows"
            )
    print(f"Migration complete. Inserted {total_inserted:,} rows.")
elif EXECUTE_MIGRATION:
    print("EXECUTE_MIGRATION is True, but there are no rows to insert.")
else:
    print("Preview only. Set EXECUTE_MIGRATION = True to insert these rows.")

Rows queued for insert: 100,362
History IDs affected:   728


,obs_time,datum,vars_id,history_id,mod_user
0,2026-04-23 02:00:00,0.0,496,2014,crmprtd
1,2026-04-23 03:00:00,0.0,496,2014,crmprtd
2,2026-04-23 04:00:00,0.0,496,2014,crmprtd
3,2026-04-23 05:00:00,0.0,496,2014,crmprtd
4,2026-04-23 06:00:00,0.0,496,2014,crmprtd
5,2026-04-23 07:00:00,0.0,496,2014,crmprtd
6,2026-04-23 08:00:00,0.0,496,2014,crmprtd
7,2026-04-23 09:00:00,0.0,496,2014,crmprtd
8,2026-04-23 10:00:00,0.0,496,2014,crmprtd
9,2026-04-23 11:00:00,0.0,496,2014,crmprtd


Inserted 10,000/100,362 recent missing rows
Inserted 20,000/100,362 recent missing rows
Inserted 30,000/100,362 recent missing rows
Inserted 40,000/100,362 recent missing rows
Inserted 50,000/100,362 recent missing rows
Inserted 60,000/100,362 recent missing rows
Inserted 70,000/100,362 recent missing rows
Inserted 80,000/100,362 recent missing rows
Inserted 90,000/100,362 recent missing rows
Inserted 100,000/100,362 recent missing rows
Inserted 100,362/100,362 recent missing rows
Migration complete. Inserted 100,362 rows.


## Verify recent window after migration

In [15]:
with engine_old.connect() as conn:
    old_recent_count = conn.execute(
        text("SELECT COUNT(*) FROM obs_raw WHERE obs_time >= :cutoff"),
        {"cutoff": cutoff},
    ).scalar()

with engine_new.connect() as conn:
    new_recent_count = conn.execute(
        text("SELECT COUNT(*) FROM obs_raw WHERE obs_time >= :cutoff"),
        {"cutoff": cutoff},
    ).scalar()

remaining_missing_count = old_recent_count - new_recent_count

print(f"Cutoff: {cutoff}")
print(f"Old DB recent obs_raw count: {old_recent_count:,}")
print(f"New DB recent obs_raw count: {new_recent_count:,}")
print(f"Remaining recent-row gap:    {remaining_missing_count:+,}")

Cutoff: 2026-04-22 00:00:00
Old DB recent obs_raw count: 795,294
New DB recent obs_raw count: 796,322
Remaining recent-row gap:    -1,028


## Some rows sanity check

In [16]:
SAMPLE_SIZE = 10
sample_query = text("""
    SELECT obs_time, datum, vars_id, history_id, mod_user
    FROM obs_raw
    WHERE history_id = :history_id
      AND vars_id = :vars_id
      AND obs_time = :obs_time
""")

if rows_to_insert.empty:
    print("No recent rows available for sanity checking.")
else:
    sample_old_rows = rows_to_insert.head(SAMPLE_SIZE).copy()

    sample_new_rows = []
    with engine_new.connect() as conn:
        for row in sample_old_rows.itertuples(index=False):
            result = conn.execute(
                sample_query,
                {
                    "history_id": row.history_id,
                    "vars_id": row.vars_id,
                    "obs_time": row.obs_time,
                },
            ).mappings().all()
            sample_new_rows.extend(result)

    sample_new_rows_df = pd.DataFrame(sample_new_rows)
    if not sample_new_rows_df.empty:
        sample_new_rows_df["obs_time"] = pd.to_datetime(sample_new_rows_df["obs_time"])

    sample_check = sample_old_rows.merge(
        sample_new_rows_df,
        on=["history_id", "vars_id", "obs_time"],
        how="left",
        suffixes=("_old", "_new"),
    )
    sample_check["found_in_new"] = sample_check["datum_new"].notna()

    print(f"Sampled {len(sample_old_rows)} rows from the recent insert set.")
    print(f"Found in new DB: {int(sample_check['found_in_new'].sum())}/{len(sample_check)}")
    display(sample_check)

Sampled 10 rows from the recent insert set.
Found in new DB: 10/10


,obs_time,datum_old,vars_id,history_id,mod_user_old,datum_new,mod_user_new,found_in_new
0,2026-04-24 01:00:00,46.2,106,9,crmprtd,46.2,metnorth,True
1,2026-04-24 02:00:00,44.0,106,9,crmprtd,44.0,metnorth,True
2,2026-04-24 03:00:00,39.0,106,9,crmprtd,39.0,metnorth,True
3,2026-04-24 04:00:00,35.2,106,9,crmprtd,35.2,metnorth,True
4,2026-04-24 05:00:00,31.1,106,9,crmprtd,31.1,metnorth,True
5,2026-04-24 06:00:00,44.5,106,9,crmprtd,44.5,metnorth,True
6,2026-04-24 07:00:00,37.3,106,9,crmprtd,37.3,metnorth,True
7,2026-04-24 08:00:00,41.7,106,9,crmprtd,41.7,metnorth,True
8,2026-04-24 01:00:00,-2.2,108,9,crmprtd,-2.2,metnorth,True
9,2026-04-24 02:00:00,-2.2,108,9,crmprtd,-2.2,metnorth,True
